# Swiss Legal Corpus — Text Characteristics Analysis

Goal: understand why BERTopic produced ~42% outliers, and how to differentiate between legal topics/subtopics.

Sections:
1. Setup & data loading
2. Text length & structural patterns
3. Cross-article cosine similarity distribution
4. Domain breakdown (length, similarity by domain)
5. Discriminative vocabulary per domain (TF-IDF)
6. Legal citation / reference patterns
7. Domain vocabulary overlap

## Section 1 — Setup & Data Loading

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter

warnings.filterwarnings('ignore')

BASE     = '../data'
BERT_DIR = '../data/bertopic'

laws_df = pd.read_csv(f'{BASE}/laws_de.csv')
laws_df['text']  = laws_df['text'].fillna('')
laws_df['title'] = laws_df['title'].fillna('')

# ── Domain assignment ────────────────────────────────────────────────────────
laws_df['law_code'] = laws_df['citation'].str.extract(r'(\d[\d\.]*\d|\d)\s*$')
laws_df['domain']   = laws_df['law_code'].str.extract(r'^(\d)', expand=False)

ABBREV_TO_DOMAIN = {
    'BV': '1', 'AIG': '1', 'VZAE': '1', 'VG': '1', 'BGG': '1',
    'PO': '1', 'BPV': '1', 'EJPD': '1', 'EDA': '1', 'NDG': '1',
    'ASG': '1', 'ISG': '1', 'VBP': '1', 'BGS': '1', 'BPR': '1',
    'RVOG': '1', 'RVOV': '1', 'GE': '1',
    'ZGB': '2', 'GBV': '2', 'GB': '2', 'ZV': '2', 'OR': '2',
    'KG': '2',  'ZPO': '2', 'ERV': '2', 'IPRG': '2', 'SV': '2',
    'HG': '2',  'DS': '2',  'PF': '2',  'URG': '2', 'VZG': '2',
    'VVG': '2', 'BGBB': '2', 'DV': '2', 'BG': '2',
    'IRSG': '3',
    'ETH': '4', 'HFKG': '4', 'FIFG': '4', 'TSV': '4', 'BBV': '4',
    'MG': '5', 'MIG': '5', 'VMDP': '5', 'VVA': '5', 'VMSV': '5',
    'WG': '5', 'BZG': '5', 'ZSV': '5', 'VBS': '5',
    'ZG': '6', 'MWSTG': '6', 'MWSTV': '6', 'DBG': '6', 'BAZG': '6', 'SVAV': '6',
    'SVG': '7', 'VRV': '7', 'SSV': '7', 'VZV': '7', 'VVV': '7',
    'EBG': '7', 'EBV': '7', 'VTS': '7', 'LFG': '7', 'LFV': '7',
    'VIL': '7', 'SSG': '7', 'VPB': '7', 'FMG': '7', 'FDV': '7',
    'FMV': '7', 'PG': '7', 'VPG': '7', 'RTVG': '7', 'RTVV': '7',
    'EMKV': '7', 'KEG': '7', 'WV': '7', 'WRG': '7',
    'HMG': '8', 'VAM': '8', 'USG': '8', 'LMVV': '8', 'LGV': '8',
    'VSFK': '8', 'ZDV': '8', 'ZDG': '8', 'VUV': '8', 'AHVG': '8',
    'AHVV': '8', 'IVG': '8', 'IVV': '8', 'BVG': '8', 'KVG': '8',
    'KVV': '8', 'KKV': '8', 'KLV': '8', 'UVG': '8', 'UVV': '8',
    'AVIG': '8', 'AVIV': '8', 'MVG': '8', 'LMG': '8', 'ATSG': '8',
    'PBG': '8', 'KV': '8', 'AV': '8', 'BSV': '8', 'VRAB': '8', 'PV': '8',
    'FINMA': '9', 'FINMAG': '9', 'AVO': '9', 'VAG': '9', 'KAG': '9',
    'FV': '9', 'FIDLEG': '9', 'FIDLEV': '9', 'FINIV': '9', 'UEV': '9',
    'VGS': '9', 'SVV': '9', 'DZV': '9', 'PSMV': '9',
}

abbrev_series = (
    laws_df.loc[laws_df['domain'].isna(), 'citation']
    .str.extract(r'([A-Z][A-Z0-9]+)\s*$')[0]
)
laws_df.loc[laws_df['domain'].isna(), 'domain'] = abbrev_series.map(ABBREV_TO_DOMAIN)
laws_df['domain'] = laws_df['domain'].fillna('?')

domain_labels = {
    '1': '1xx: State & Constitution',
    '2': '2xx: Private Law & Procedure',
    '3': '3xx: Criminal Law',
    '4': '4xx: Education & Culture',
    '5': '5xx: Defense & Civil Protection',
    '6': '6xx: Finance & Tax',
    '7': '7xx: Transport & Telecom',
    '8': '8xx: Health, Labor & Social',
    '9': '9xx: Economic Law',
    '?': '?: Still unknown',
}
DOMAIN_LIST = sorted(d for d in domain_labels if d != '?')

print(f'Loaded {len(laws_df):,} articles, {laws_df["law_code"].nunique()} SR law codes')
print()
print('Domain distribution:')
dist = laws_df['domain'].value_counts().sort_index()
for d, n in dist.items():
    print(f'  {domain_labels.get(d,d):35s}  {n:6,}  ({100*n/len(laws_df):.1f}%)')
laws_df.head(3)

## Section 2 — Text Length & Structural Patterns

In [ ]:
laws_df['n_words'] = laws_df['text'].str.split().str.len()

print('=== Word count per article ===')
print(laws_df['n_words'].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

word_cap = laws_df['n_words'].quantile(0.99)
axes[0].hist(laws_df['n_words'].clip(upper=word_cap), bins=80, color='steelblue', edgecolor='white')
axes[0].axvline(laws_df['n_words'].median(), color='red', linestyle='--',
                label=f'median={laws_df["n_words"].median():.0f}')
axes[0].set_xlabel('Word count (clipped at 99th pct)')
axes[0].set_ylabel('Articles')
axes[0].set_title('Article length — words')
axes[0].legend()

# What are the most common first words? (reveals formulaic openings)
first_words = laws_df['text'].str.strip().str.split().str[0].str.lower().dropna()
top_first = first_words.value_counts().head(15)
axes[1].bar(top_first.index, top_first.values, color='darkorange', edgecolor='white')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Count')
axes[1].set_title('Most common first words (reveals formulaic structure)')

plt.tight_layout()
plt.show()

# How many very short articles?
for thresh in [10, 20, 50, 100]:
    n = (laws_df['n_words'] < thresh).sum()
    print(f'Articles with <{thresh} words: {n:,} ({100*n/len(laws_df):.1f}%)')

# Top 3-word opening phrases — show how formulaic the corpus is
def first_ngram(text, n=3):
    words = text.strip().lower().split()
    return ' '.join(words[:n]) if len(words) >= n else None

opening_trigrams = laws_df['text'].apply(first_ngram).dropna()
top_trigrams = opening_trigrams.value_counts().head(20)
pct = top_trigrams.sum() / len(opening_trigrams) * 100
print(f'\nTop 20 opening trigrams cover {pct:.1f}% of all articles — indicates high structural uniformity')
print('\nTop 20 opening 3-word phrases:')
for phrase, count in top_trigrams.items():
    print(f'  {count:5d}  "{phrase}"')

## Section 3 — Cross-article Cosine Similarity Distribution

Key question: are articles semantically similar to each other on average?
If mean pairwise cosine sim is high, it explains why HDBSCAN finds ~42% outliers — no clear density peaks.

In [ ]:
import os

embs_path = f'{BERT_DIR}/corpus_embs.npy'
if os.path.exists(embs_path):
    corpus_embs = np.load(embs_path)
    print(f'Loaded embeddings: {corpus_embs.shape}')
else:
    print('No cached embeddings found. Run 2026-03-12-bertopic.ipynb sec5 first.')

In [ ]:
# Sample 2000 articles for pairwise cosine similarity (full pairwise on 50k+ is too expensive)
SAMPLE_N = 2000
rng = np.random.default_rng(42)
idx = rng.choice(len(corpus_embs), size=SAMPLE_N, replace=False)
sample_embs = corpus_embs[idx]  # already L2-normalized

# Cosine sim matrix: since embeddings are normalized, dot product = cosine sim
sim_matrix = sample_embs @ sample_embs.T  # [2000, 2000]

# Extract upper triangle (excluding diagonal)
upper_idx = np.triu_indices(SAMPLE_N, k=1)
pairwise_sims = sim_matrix[upper_idx]

print(f'Pairwise cosine similarities ({SAMPLE_N} sample, {len(pairwise_sims):,} pairs):')
print(f'  mean:   {pairwise_sims.mean():.4f}')
print(f'  median: {np.median(pairwise_sims):.4f}')
print(f'  std:    {pairwise_sims.std():.4f}')
print(f'  min:    {pairwise_sims.min():.4f}')
print(f'  max:    {pairwise_sims.max():.4f}')
print(f'  % pairs with sim > 0.8: {(pairwise_sims > 0.8).mean()*100:.1f}%')
print(f'  % pairs with sim > 0.9: {(pairwise_sims > 0.9).mean()*100:.1f}%')

plt.figure(figsize=(10, 4))
plt.hist(pairwise_sims, bins=100, color='steelblue', edgecolor='white')
plt.axvline(pairwise_sims.mean(), color='red', linestyle='--', label=f'mean={pairwise_sims.mean():.3f}')
plt.axvline(np.median(pairwise_sims), color='orange', linestyle='--', label=f'median={np.median(pairwise_sims):.3f}')
plt.xlabel('Cosine similarity')
plt.ylabel('Pair count')
plt.title(f'Pairwise cosine similarity distribution (sample n={SAMPLE_N})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Nearest-neighbour similarity: for each article, what's the similarity to its closest neighbour?
# More informative than mean pairwise — tells us local density
np.fill_diagonal(sim_matrix, -1)  # exclude self
nn_sims = sim_matrix.max(axis=1)

print(f'Nearest-neighbour cosine similarity:')
print(f'  mean:   {nn_sims.mean():.4f}')
print(f'  median: {np.median(nn_sims):.4f}')
print(f'  % with NN sim < 0.7: {(nn_sims < 0.7).mean()*100:.1f}%  (truly isolated docs)')
print(f'  % with NN sim > 0.9: {(nn_sims > 0.9).mean()*100:.1f}%  (near-duplicate or very tight clusters)')

plt.figure(figsize=(10, 4))
plt.hist(nn_sims, bins=80, color='darkorange', edgecolor='white')
plt.axvline(np.median(nn_sims), color='red', linestyle='--', label=f'median={np.median(nn_sims):.3f}')
plt.xlabel('Cosine similarity to nearest neighbour')
plt.ylabel('Articles')
plt.title('Nearest-neighbour similarity (local density proxy)')
plt.legend()
plt.tight_layout()
plt.show()

## Section 4 — Domain Breakdown

In [ ]:
# Article count and median length per domain
domain_stats = (
    laws_df.groupby('domain')['n_words']
    .agg(['count', 'median', 'mean'])
    .round(1)
    .rename(columns={'count': 'n_articles', 'median': 'median_words', 'mean': 'mean_words'})
)
domain_stats.index = [domain_labels.get(d, d) for d in domain_stats.index]
print(domain_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
domains_ordered = [domain_labels.get(d, d) for d in sorted(laws_df['domain'].unique())]

n_articles = [laws_df[laws_df['domain'] == d].shape[0]
              for d in sorted(laws_df['domain'].unique())]
med_words   = [laws_df[laws_df['domain'] == d]['n_words'].median()
               for d in sorted(laws_df['domain'].unique())]

axes[0].barh(domains_ordered, n_articles, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of articles')
axes[0].set_title('Article count by domain')

axes[1].barh(domains_ordered, med_words, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Median word count')
axes[1].set_title('Median article length by domain')

plt.tight_layout()
plt.show()

In [ ]:
# Within-domain vs cross-domain cosine similarity
# Use the same sample, compare sim between same-domain pairs vs different-domain pairs
sample_domains = laws_df.iloc[idx]['domain'].values

np.fill_diagonal(sim_matrix, 0)  # reset diagonal
upper_idx = np.triu_indices(SAMPLE_N, k=1)
sims   = sim_matrix[upper_idx]
d_i    = sample_domains[upper_idx[0]]
d_j    = sample_domains[upper_idx[1]]
same   = d_i == d_j

print(f'Within-domain pairs:  mean sim = {sims[same].mean():.4f}  (n={same.sum():,})')
print(f'Cross-domain pairs:   mean sim = {sims[~same].mean():.4f}  (n={(~same).sum():,})')
print(f'Gap: {sims[same].mean() - sims[~same].mean():.4f}')

plt.figure(figsize=(10, 4))
plt.hist(sims[same],  bins=80, alpha=0.6, color='steelblue',   label='Same domain',  density=True)
plt.hist(sims[~same], bins=80, alpha=0.6, color='darkorange',  label='Cross domain', density=True)
plt.xlabel('Cosine similarity')
plt.ylabel('Density')
plt.title('Within-domain vs cross-domain cosine similarity')
plt.legend()
plt.tight_layout()
plt.show()

## Section 5 — Discriminative Vocabulary per Domain (TF-IDF)

The core problem: legalese (Absatz, gemäss, Artikel, Bundesrat…) is shared across all domains.
To find what **characterises** each domain, we treat each domain as one big document and compute TF-IDF.
High TF-IDF term in domain X = appears often in X but rarely in other domains → true topic marker.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build one mega-document per domain
domain_docs = {}
for d in sorted(laws_df['domain'].unique()):
    texts = laws_df[laws_df['domain'] == d]['text'].tolist()
    domain_docs[d] = ' '.join(texts)

domain_keys = sorted(domain_docs.keys())
corpus = [domain_docs[d] for d in domain_keys]

# min_df=2 to filter noise, max_df=0.85 to suppress cross-domain stopwords
tfidf = TfidfVectorizer(min_df=2, max_df=0.85, max_features=30_000,
                         token_pattern=r'(?u)\b[a-zA-ZäöüÄÖÜß]{4,}\b')
tfidf_matrix = tfidf.fit_transform(corpus)  # [n_domains, n_terms]
vocab = np.array(tfidf.get_feature_names_out())

TOP_N = 15
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for i, (d, ax) in enumerate(zip(domain_keys, axes)):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:TOP_N]
    top_terms = vocab[top_idx]
    top_scores = scores[top_idx]
    label = domain_labels.get(d, d)

    ax.barh(top_terms[::-1], top_scores[::-1], color='steelblue', edgecolor='white')
    ax.set_title(label, fontsize=9)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_xlabel('TF-IDF score', fontsize=8)

# Hide unused subplots
for ax in axes[len(domain_keys):]:
    ax.set_visible(False)

plt.suptitle('Top 15 discriminative terms per domain (TF-IDF)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Section 6 — Legal Citation / Reference Patterns

Legal articles reference other provisions using both abbreviated (`Art.`, `Abs.`, `lit.`) and
full German forms (`Artikel`, `Absatz`, `Buchstabe`). Articles that cite the same laws
cluster naturally — citations are a topic signal independent of vocabulary.

In [ ]:
# Extract citation-like patterns — using both abbreviated and full German forms
pat_art = re.compile(r'\b(Art\.?|Artikel)\s*\d+',       re.IGNORECASE)  # Art. 12 / Artikel 12
pat_par = re.compile(r'§\s*\d+')                                          # § 45
pat_abs = re.compile(r'\b(Abs\.?|Absatz)\s*\d+',        re.IGNORECASE)  # Abs. 3 / Absatz 3
pat_lit = re.compile(r'\b(lit\.?|Bst\.?|Buchstabe)\s*[a-z]', re.IGNORECASE)  # lit. a / Bst. a / Buchstabe a

def count_matches(text, pattern):
    return len(pattern.findall(text))

laws_df['n_art'] = laws_df['text'].apply(lambda t: count_matches(t, pat_art))
laws_df['n_par'] = laws_df['text'].apply(lambda t: count_matches(t, pat_par))
laws_df['n_abs'] = laws_df['text'].apply(lambda t: count_matches(t, pat_abs))
laws_df['n_lit'] = laws_df['text'].apply(lambda t: count_matches(t, pat_lit))
laws_df['n_refs'] = laws_df['n_art'] + laws_df['n_par'] + laws_df['n_abs'] + laws_df['n_lit']

print('=== Citation counts across corpus ===')
print(f"Art./Artikel references:  {laws_df['n_art'].sum():,}  (median per article: {laws_df['n_art'].median():.1f})")
print(f"§ references:             {laws_df['n_par'].sum():,}  (median per article: {laws_df['n_par'].median():.1f})")
print(f"Abs./Absatz references:   {laws_df['n_abs'].sum():,}  (median per article: {laws_df['n_abs'].median():.1f})")
print(f"lit./Bst./Buchstabe:      {laws_df['n_lit'].sum():,}  (median per article: {laws_df['n_lit'].median():.1f})")
print(f"\nArticles with 0 references: {(laws_df['n_refs'] == 0).sum():,} ({(laws_df['n_refs'] == 0).mean()*100:.1f}%)")
print(f"Articles with ≥1 reference: {(laws_df['n_refs'] >= 1).sum():,} ({(laws_df['n_refs'] >= 1).mean()*100:.1f}%)")

# Citation density per domain
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

domains_ordered = [domain_labels.get(d, d) for d in sorted(laws_df['domain'].unique())]
ref_by_domain = [laws_df[laws_df['domain'] == d]['n_refs'].median()
                 for d in sorted(laws_df['domain'].unique())]
axes[0].barh(domains_ordered, ref_by_domain, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Median reference count per article')
axes[0].set_title('Citation density by domain')

ref_cap = laws_df['n_refs'].quantile(0.99)
axes[1].hist(laws_df['n_refs'].clip(upper=ref_cap), bins=60, color='darkorange', edgecolor='white')
axes[1].axvline(laws_df['n_refs'].median(), color='red', linestyle='--',
                label=f'median={laws_df["n_refs"].median():.1f}')
axes[1].set_xlabel('Reference count (clipped at 99th pct)')
axes[1].set_ylabel('Articles')
axes[1].set_title('References per article')
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 7 — Domain Vocabulary Overlap

How much of each domain's vocabulary is shared with other domains?
High Jaccard overlap → domains are hard to separate with vocabulary-based methods.
Low overlap → domains have distinct lexicons → topic models should work well there.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

TOP_VOCAB = 1000  # use top-1000 words per domain for comparison

# Build per-domain word sets (top N by frequency, min 4 chars to exclude noise)
cv = CountVectorizer(max_features=50_000, token_pattern=r'(?u)\b[a-zA-ZäöüÄÖÜß]{4,}\b')
cv.fit([domain_docs[d] for d in domain_keys])
vocab_all = np.array(cv.get_feature_names_out())

domain_top_sets = {}
for d in domain_keys:
    counts = cv.transform([domain_docs[d]]).toarray().flatten()
    top_idx = counts.argsort()[::-1][:TOP_VOCAB]
    domain_top_sets[d] = set(vocab_all[top_idx])

# Jaccard similarity matrix
n = len(domain_keys)
jaccard = np.zeros((n, n))
for i, di in enumerate(domain_keys):
    for j, dj in enumerate(domain_keys):
        inter = len(domain_top_sets[di] & domain_top_sets[dj])
        union = len(domain_top_sets[di] | domain_top_sets[dj])
        jaccard[i, j] = inter / union if union > 0 else 0.0

labels_short = [domain_labels.get(d, d).split(':')[0] for d in domain_keys]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(jaccard, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Jaccard similarity')
ax.set_xticks(range(n)); ax.set_xticklabels(labels_short, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(n)); ax.set_yticklabels(labels_short, fontsize=9)
ax.set_title(f'Domain vocabulary overlap (Jaccard, top-{TOP_VOCAB} words per domain)', fontsize=11)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{jaccard[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if jaccard[i, j] > 0.6 else 'black')

plt.tight_layout()
plt.show()

# Print most / least separable domain pairs
pairs = [(jaccard[i, j], domain_labels.get(domain_keys[i], domain_keys[i]),
          domain_labels.get(domain_keys[j], domain_keys[j]))
         for i in range(n) for j in range(i+1, n)]
pairs.sort(key=lambda x: x[0])

print('Most separable domain pairs (lowest Jaccard):')
for sim, a, b in pairs[:3]:
    print(f'  {sim:.3f}  {a}  ↔  {b}')

print('\nHardest domain pairs to separate (highest Jaccard):')
for sim, a, b in pairs[-3:]:
    print(f'  {sim:.3f}  {a}  ↔  {b}')

## Section 8 — TF-IDF Embedding Baseline

Build a vocabulary-based "embedding" using only the discriminative terms from Section 5.
Each article is represented as a TF-IDF vector over the top-N terms per domain.

This gives a clean 3-way comparison:
- **TF-IDF embedding** (this section) — vocabulary signal only
- **Neural embedding baseline** — gap = 0.0208 (Section 4)
- **Fine-tuned neural embedding** — after training notebook

Metrics: within/cross-domain cosine gap + kNN accuracy on gold test set.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import normalize

# ── Step 1: collect discriminative vocabulary ─────────────────────────────────
# Re-use the tfidf_matrix and vocab from Section 5
# Take top-K terms per domain, union them → discriminative vocab
TOP_K_PER_DOMAIN = 100  # top 100 discriminative terms per domain

disc_vocab = set()
for i, d in enumerate(domain_keys):
    if d == '?':
        continue
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:TOP_K_PER_DOMAIN]
    disc_vocab.update(vocab[top_idx])

disc_vocab = sorted(disc_vocab)
print(f'Discriminative vocabulary size: {len(disc_vocab):,} terms')
print(f'(from top-{TOP_K_PER_DOMAIN} terms × {len([d for d in domain_keys if d != "?"])} domains)')
print(f'Sample terms: {disc_vocab[:10]}')

In [ ]:
# ── Step 2: fit TF-IDF vectorizer restricted to discriminative vocab ──────────
disc_tfidf = TfidfVectorizer(
    vocabulary=disc_vocab,
    token_pattern=r'(?u)\b[a-zA-ZäöüÄÖÜß]{4,}\b',
    sublinear_tf=True,   # log(1+tf) — dampens very frequent terms
)
disc_tfidf.fit(laws_df['text'].tolist())

# Encode full corpus → sparse matrix, then L2-normalise for cosine similarity
print('Encoding full corpus...')
X_full = disc_tfidf.transform(laws_df['text'].tolist())
X_full_norm = normalize(X_full, norm='l2')
print(f'Embedding matrix: {X_full_norm.shape}  (sparse)')

In [ ]:
# ── Step 3: within vs cross-domain cosine gap ────────────────────────────────
SAMPLE_N = 2000
rng3 = np.random.default_rng(99)
# Sample only labeled articles for a fair comparison
labeled_mask = laws_df['domain'] != '?'
labeled_idx  = np.where(labeled_mask)[0]
sample_idx   = rng3.choice(labeled_idx, size=min(SAMPLE_N, len(labeled_idx)), replace=False)

X_sample      = X_full_norm[sample_idx].toarray()
labels_sample = laws_df['domain'].iloc[sample_idx].values

sim_matrix = X_sample @ X_sample.T
upper = np.triu_indices(len(sample_idx), k=1)
sims  = sim_matrix[upper]
same  = labels_sample[upper[0]] == labels_sample[upper[1]]

within_mean = sims[same].mean()
cross_mean  = sims[~same].mean()
gap         = within_mean - cross_mean

print('=== TF-IDF Embedding — domain separation ===')
print(f'Within-domain mean sim:  {within_mean:.4f}')
print(f'Cross-domain  mean sim:  {cross_mean:.4f}')
print(f'Gap:                     {gap:.4f}')
print(f'  (neural embedding baseline gap was 0.0208)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sims[same],  bins=80, alpha=0.6, color='steelblue',  label='Same domain',  density=True)
ax.hist(sims[~same], bins=80, alpha=0.6, color='darkorange', label='Cross domain', density=True)
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Density')
ax.set_title('TF-IDF embedding — within vs cross-domain similarity')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 4: kNN classification on gold test set ───────────────────────────────
gold_df = pd.read_csv(f'{BASE}/gold_test_5000.csv')

# Train on labeled articles NOT in gold test
gold_citations = set(gold_df['citation'].tolist())
train_mask = labeled_mask & ~laws_df['citation'].isin(gold_citations)
X_train    = X_full_norm[train_mask]
y_train    = laws_df.loc[train_mask, 'domain'].values

# Encode gold test
X_gold = disc_tfidf.transform(gold_df['text'].tolist())
X_gold = normalize(X_gold, norm='l2')
y_gold = gold_df['domain'].values

print(f'Train: {X_train.shape[0]:,}  |  Test: {X_gold.shape[0]:,}')

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_gold)

domain_label_list = [domain_labels.get(d, d) for d in sorted(set(y_gold))]
print('\n=== 5-NN Classification Report — TF-IDF embedding ===')
print(classification_report(y_gold, y_pred, target_names=domain_label_list, digits=3))

In [ ]:
# ── Step 5: summary comparison table ─────────────────────────────────────────
print('=== Embedding comparison summary ===')
print(f'{"Method":<35}  {"Within/Cross gap":>18}  {"Notes"}')
print('-' * 75)
print(f'{"Neural (baseline, untuned)":<35}  {"0.0208":>18}  multilingual-e5, no fine-tuning')
print(f'{"TF-IDF discriminative vocab":<35}  {gap:>18.4f}  top-{TOP_K_PER_DOMAIN} terms/domain, kNN above')
print(f'{"Fine-tuned neural":<35}  {"(run training nb)":>18}  target: >>0.02')
print()
print('Interpretation:')
print('  gap > 0.10  → strong vocabulary signal, TF-IDF alone is competitive')
print('  gap < 0.05  → vocabulary insufficient, fine-tuning is essential')

## Section 9 — Manual Inspection of Predicted Domain Labels

Sample ~50 articles from the classified unknowns (stratified by predicted domain).
Read the citation + title + first 200 chars of text and judge: does the predicted domain make sense?

In [ ]:
unknown_clf = pd.read_csv(f'{BASE}/unknown_classified.csv')
unknown_clf['domain_pred'] = unknown_clf['domain_pred'].astype(str)

rng_inspect = np.random.default_rng(42)
N_PER_DOMAIN = 6

rows = []
for d in DOMAIN_LIST:
    df_d = unknown_clf[unknown_clf['domain_pred'] == d]
    k    = min(N_PER_DOMAIN, len(df_d))
    if k == 0:
        continue
    rows.append(df_d.sample(k, random_state=int(rng_inspect.integers(1000))))

inspect_df = pd.concat(rows).sort_values('domain_pred').reset_index(drop=True)
print(f'Inspecting {len(inspect_df)} articles across {inspect_df["domain_pred"].nunique()} domains\n')

for _, row in inspect_df.iterrows():
    label = domain_labels.get(row['domain_pred'], row['domain_pred'])
    title = str(row.get('title', '') or '').strip()
    text  = str(row.get('text',  '') or '').strip()
    conf  = row.get('domain_conf', float('nan'))
    print(f"{'─'*80}")
    print(f"Domain  : {label}  (conf={conf:.3f})")
    print(f"Citation: {row['citation']}")
    if title:
        print(f"Title   : {title}")
    print(f"Text    :\n{text}")
    print()

In [ ]:
mixed = unknown_clf[unknown_clf['domain_conf'] < 1.0].copy()
print(f'Mixed-confidence articles (conf < 1.0): {len(mixed):,}  '
      f'({100*len(mixed)/len(unknown_clf):.1f}% of unknowns)\n')

print('Domain distribution of mixed predictions:')
print(mixed['domain_pred'].value_counts().sort_index().to_string())
print()

sample_low = mixed.nsmallest(10, 'domain_conf')
for _, row in sample_low.iterrows():
    label = domain_labels.get(str(row['domain_pred']), str(row['domain_pred']))
    title = str(row.get('title', '') or '').strip()
    text  = str(row.get('text',  '') or '').strip()
    print(f"{'─'*80}")
    print(f"Domain  : {label}  (conf={row['domain_conf']:.3f})")
    print(f"Citation: {row['citation']}")
    if title:
        print(f"Title   : {title}")
    print(f"Text    :")
    print(text)
    print()